## Problem 1

### Generating Random Boolean Functions
The Deutsch–Jozsa algorithm is designed to work with functions that accept a fixed number of Boolean inputs and return a single Boolean output. Each function is guaranteed to be either constant (always returns False or always returns True) or balanced (returns True for exactly half of the possible input combinations). Write a Python function random_constant_balanced that returns a randomly chosen function from the set of constant or balanced functions taking four Boolean arguments as inputs.

Concept notes:
- A constant function returns the same output for every input.
- A balanced function returns True for exactly half of all inputs and False for the other half.
- With 4 input bits, there are 16 possible inputs, so a balanced function has 8 True outputs and 8 False outputs.

In [12]:
import random
from itertools import product

def random_constant_balanced():
    """
    Return a random 4-input Boolean function f(a, b, c, d) that is promised to be
    either constant or balanced.
    """
    # Build the full 4-bit input space: 2^4 = 16 tuples.
    inputs = list(product([False, True], repeat=4))

    # Randomly choose which promised family to sample from.
    if random.choice(["constant", "balanced"]) == "constant":
        value = random.choice([False, True])

        # Constant function ignores inputs and returns one fixed value.
        def f(a, b, c, d):
            return value

    else:
        # Balanced function is True on exactly half the input space.
        true_set = set(random.sample(inputs, k=len(inputs) // 2))

        def f(a, b, c, d):
            return (a, b, c, d) in true_set

    return f


# Quick sanity check: evaluate one random promised function.
f = random_constant_balanced()
print(f(False, False, False, False))

False


## Problem 2

### Classical Testing for Function Type
Deutsch's algorithm is designed to demonstrate a potential advantage of quantum computing over classical computation. To understand this advantage, we must first understand the classical cost of solving the underlying problem. Write a Python function determine_constant_balanced that takes as input a function f, as defined in Problem 1. The function should analyze f and return the string "constant" or "balanced" depending on whether the function is constant or balanced. Write a brief note on the efficiency of your solution. What is the maximum number of times you need to call f to be 100% certain whether it is constant or balanced?

Concept notes:
- Under the promise, f is only one of two types: constant or balanced.
- The algorithm can stop early once it has enough evidence to reject one type.
- Worst-case classical certainty for a 4-bit promised function is 9 evaluations.

In [8]:
def determine_constant_balanced(f):
    """
    Classify a promised 4-input Boolean function as "constant" or "balanced".
    """
    true_count = 0
    false_count = 0

    # Evaluate the function over the full input space, stopping as soon as possible.
    for a, b, c, d in product([False, True], repeat=4):
        result = bool(f(a, b, c, d))

        if result:
            true_count += 1
        else:
            false_count += 1

        # A balanced 4-bit promised function cannot exceed 8 of one output value.
        if true_count > 8 or false_count > 8:
            return "constant"

        # Seeing both True and False immediately rules out constant.
        if true_count > 0 and false_count > 0:
            return "balanced"

    # If every observed output is identical, the function is constant.
    return "constant"


# Example usage and worst-case query statement.
f = random_constant_balanced()
kind = determine_constant_balanced(f)
print(f"Function type: {kind}")
print("Worst-case classical calls needed for certainty: 9")

Function type: balanced
Worst-case classical calls needed for certainty: 9


## Problem 3

### Quantum Oracles
Deutsch's algorithm is the simplest example of a quantum algorithm using superposition to determine a global property of a function with a single evaluation. In the single-input case, there are four possible Boolean functions. Using Qiskit, create the appropriate quantum oracles for each of the possible single-Boolean-input functions used in Deutsch's algorithm. Demonstrate their use and explain how each oracle implements its corresponding function.

Concept notes:
- A Deutsch oracle implements the reversible map |x,y> -> |x, y xor f(x)|.
- Reversibility is required for valid quantum operations.
- Different gate patterns represent the four possible single-bit Boolean functions.

In [9]:
"""
Quantum oracles for Deutsch's algorithm:
- f(x)=0: identity
- f(x)=1: X on target
- f(x)=x: CNOT
- f(x)=not x: X, CNOT, X
"""

from qiskit import QuantumCircuit


def oracle_f0():
    """f(x)=0: |x, y> -> |x, y>"""
    qc = QuantumCircuit(2, name="Uf0")
    # Identity operation: y is unchanged for both x=0 and x=1.
    return qc


def oracle_f1():
    """f(x)=1: |x, y> -> |x, y xor 1>"""
    qc = QuantumCircuit(2, name="Uf1")
    # Always flip target qubit because f(x)=1 for all x.
    qc.x(1)
    return qc


def oracle_fx():
    """f(x)=x: |x, y> -> |x, y xor x>"""
    qc = QuantumCircuit(2, name="Ufx")
    # CNOT flips y iff control x is 1.
    qc.cx(0, 1)
    return qc


def oracle_fnotx():
    """f(x)=not x: |x, y> -> |x, y xor (not x)>"""
    qc = QuantumCircuit(2, name="Ufnotx")
    # Convert control-on-0 into control-on-1 with X gates around CNOT.
    qc.x(0)
    qc.cx(0, 1)
    qc.x(0)
    return qc


# Build and display all four possible single-bit Deutsch oracles.
oracles = {
    "f(x)=0": oracle_f0(),
    "f(x)=1": oracle_f1(),
    "f(x)=x": oracle_fx(),
    "f(x)=not x": oracle_fnotx(),
}

for name, oracle in oracles.items():
    print(f"\n{name}")
    print(oracle.draw(output="text"))


f(x)=0
     
q_0: 
     
q_1: 
     

f(x)=1
          
q_0: ─────
     ┌───┐
q_1: ┤ X ├
     └───┘

f(x)=x
          
q_0: ──■──
     ┌─┴─┐
q_1: ┤ X ├
     └───┘

f(x)=not x
     ┌───┐     ┌───┐
q_0: ┤ X ├──■──┤ X ├
     └───┘┌─┴─┐└───┘
q_1: ─────┤ X ├─────
          └───┘     


## Problem 4

### Deutsch's Algorithm with Qiskit
Use Qiskit to design a quantum circuit that solves Deutsch's problem for a function with a single Boolean input. Implement the necessary circuit and demonstrate its use with each of the quantum oracles from Problem 3. Describe how the interference pattern produced by the circuit allows you to determine whether the function is constant or balanced using only one query to the oracle.

## Problem 5

### Scaling to the Deutsch–Jozsa Algorithm
The Deutsch–Jozsa algorithm generalizes Deutsch's approach to functions with several input bits. Use Qiskit to create a quantum circuit that can handle the four-bit functions generated in Problem 1. Explain how the classical function is encoded as a quantum oracle, and demonstrate the use of your circuit on both of the constant functions and any two balanced functions of your choosing. Show that the circuit correctly identifies the type of each function.